# EasyRAG Code Map And Runtime Flow

## What you will do

- trace which modules own the main RAG stage transitions
- inspect the public method signatures that connect those modules
- compare the conceptual learning path with the actual runtime entry points

## Why this matters

A tutorial is easier to trust when it lines up with the implementation. This notebook reconnects the teaching path to the code paths that actually execute in EasyRAG.


## Source code anchors

- `easyrag.rag.orchestrator`
- `easyrag.rag.types`
- `easyrag.rag.indexing.pipeline`
- `easyrag.rag.retrieval.pipeline`
- `easyrag.rag.generation.pipeline`
- `docs/engineering/20-code-map.md`


In [ ]:
# ruff: noqa: F401
import inspect
import re
import tempfile
from pathlib import Path
from pprint import pprint

from easyrag.rag import AnswerParam, EasyRAG, QueryParam
from easyrag.rag.providers import can_use_openai_compatible_models

REPO_ROOT = Path.cwd()
module_map = {
    "Public orchestrator": REPO_ROOT / "easyrag/rag/orchestrator.py",
    "Shared types": REPO_ROOT / "easyrag/rag/types.py",
    "Indexing pipeline": REPO_ROOT / "easyrag/rag/indexing/pipeline.py",
    "Retrieval pipeline": REPO_ROOT / "easyrag/rag/retrieval/pipeline.py",
    "Generation pipeline": REPO_ROOT / "easyrag/rag/generation/pipeline.py",
    "Engineering code map": REPO_ROOT / "docs/engineering/20-code-map.md",
}


## Deterministic path

We begin by reading the files as files. That keeps the notebook honest about where each stage transition lives.


In [ ]:
for label, path in module_map.items():
    text = path.read_text()
    print(f"=== {label}: {path.relative_to(REPO_ROOT)} ===")
    if path.suffix == ".py":
        names = re.findall(r"^(?:async\s+def|def|class)\s+([A-Za-z0-9_]+)", text, flags=re.MULTILINE)
        pprint(names[:12])
    else:
        print("\n".join(text.strip().splitlines()[:12]))
    print()


## Output inspection

The next cell focuses on the public surface. These signatures are the shortest path from the notebook story to the actual runtime story.


In [ ]:
signatures = {
    "EasyRAG.__init__": str(inspect.signature(EasyRAG.__init__)),
    "EasyRAG.ainsert": str(inspect.signature(EasyRAG.ainsert)),
    "EasyRAG.aquery": str(inspect.signature(EasyRAG.aquery)),
    "EasyRAG.aanswer": str(inspect.signature(EasyRAG.aanswer)),
    "EasyRAG.get_stats": str(inspect.signature(EasyRAG.get_stats)),
}
pprint(signatures)

stage_ownership = {
    "loading_and_prepare": [
        "easyrag.rag.orchestrator.EasyRAG.prepare_documents",
        "easyrag.rag.indexing.prepare.prepare_documents_for_insert",
    ],
    "indexing": [
        "easyrag.rag.indexing.chunking_core",
        "easyrag.rag.indexing.pipeline.build_insert_payloads",
        "easyrag.rag.indexing.pipeline.ingest_documents",
    ],
    "retrieval": [
        "easyrag.rag.retrieval.preprocess.QueryPreprocessor.prepare",
        "easyrag.rag.retrieval.pipeline.execute_query",
    ],
    "generation": [
        "easyrag.rag.generation.selection.select_answer_citations",
        "easyrag.rag.generation.packing.build_context_block",
        "easyrag.rag.generation.pipeline.generate_answer",
    ],
    "evaluation": [
        "easyrag.rag.evaluation.runner.evaluate_retrieval",
        "easyrag.rag.evaluation.runner.evaluate_answers",
    ],
}
pprint(stage_ownership)


## Provider-backed path

When provider-backed defaults are available, `EasyRAG` wires those callables into the orchestrator at construction time. The next cell prints that wiring instead of running a full end-to-end query.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    try:
        provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-code-map-demo")
        wiring = {
            "llm_model_func": getattr(provider_rag.llm_model_func, "__name__", type(provider_rag.llm_model_func).__name__) if provider_rag.llm_model_func else None,
            "query_model_func": getattr(provider_rag.query_model_func, "__name__", type(provider_rag.query_model_func).__name__) if provider_rag.query_model_func else None,
            "embedding_func": getattr(provider_rag.embedding_func, "__name__", type(provider_rag.embedding_func).__name__) if provider_rag.embedding_func else None,
            "reranker_func": getattr(provider_rag.reranker_func, "__name__", type(provider_rag.reranker_func).__name__) if provider_rag.reranker_func else None,
            "answer_model_func": getattr(provider_rag.answer_model_func, "__name__", type(provider_rag.answer_model_func).__name__) if provider_rag.answer_model_func else None,
        }
        pprint(wiring)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        provider_tmp.cleanup()


## What changed and why

The conceptual stage order from the docs still holds, but now you can point at the owning modules. `orchestrator.py` owns the public lifecycle. `types.py` defines the data contracts. The indexing, retrieval, generation, and evaluation packages own the stage-specific logic behind those public methods.


## Next steps

- Continue with [08_02_local_vs_production_backends.ipynb](08_02_local_vs_production_backends.ipynb) if you want to compare runtime backends after this code-oriented walkthrough.
- Read [08-system-architecture-overview.md](../../docs/08-system-architecture-overview.md) for the chapter-level architecture map.
- Keep [20-code-map.md](../../docs/engineering/20-code-map.md) nearby as the lightweight engineering reference.
